# 6 - Information Retrieval
## CSCI E-108
### Steve Elston

# Introduction

Information retrieval is a core data mining methodology. For example, document search is an essential component of solutions like web search, recommender systems and RAG algorithms for constraining results from large language models.  

This notebook contains a simple example of vector document search of a dataset containing queries and passages. Our goal is the use approximate nearest neighbor search to find the most relevant passages for each of the queries. The search approach used in this notebook is deliberately restricted. A real-world solution would be considerably more complex. In this case, the search is performed using several approaches:    
1. An approximate dense vector search using a scaleable index.
2. A sparse search using an inverted index using BM25 reweighting of terms.
3. A hybrid search using scores from both the dense vector search and tbe spares term serach.
4. A reranking or refinement step using a more exact, but slower algorithm, applied the results of the hybrid search.   

As one might expect, production quality search systems require a great deal more sophistication than is applied here. Further, we pursue only one of many possible architectures to this problem.

The approach we use for retrieval here is unsupervised. The ranking information is used only for evaluation.

## Instructions   
You will execute the code in this notebook and answer the questions in the exercise. **Provide short succinct answers that are one or a few sentences.** Do not generate answers with AI.   

> **Attribution:** Much of the code in this notebook was generated by or modified from code generated using Google Gemini.     

## Setup

In this section, you will perform the setup steps required to execute the remainder of this notebook. This notebook is intended to run in Google Colab using a GPU. If you are not familiar with working in Colab you can find a [quick start guide here](https://docs.google.com/document/d/1afPjc4IaeZzIqUAX20uBEk3Dt41pAP0Ebkpd53EJTaE/edit?tab=t.0).   

To configure the colab runtime environment click the Change runtime type under the Runtime menu tab. Select a GPU type for execution.   

### Package installation and imports

As a first step, install the required packages by executing the code in the cell below.  

> **Note 1:** Execution of this notebook using A100 GPU on Google Colab Pro+ requires several hours of wall-clock time.  
>
> **Note 2:** There are multiple possible version conflicts which arise when creating an environment where FAISS runs. You may well see errors from the pip installer. If this occurs you need to click `Restart session` under the Runtime menu tab and execute the code in the cell below again.

Now, execute the code in the cell below to install the required packages

In [ ]:
!pip install -qU \
    sentence-transformers \
    torch==2.6.0 \
    pandas==2.2.2 \
    datasets==2.12.0 \
    pyarrow<20.0.0a0\
    numpy==2.2

#!pip install rank-bm25
!pip install bm25s[full]
!pip install faiss-gpu-cu12
!pip install pinecone-text
!pip install ir_datasets

Next, execute the code below to import the required packages.  

In [ ]:
import numpy as np
import pandas as pd
import faiss
import bm25s
#from rank_bm25 import BM25Okapi

import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import torch
import textwrap
import matplotlib.pyplot as plt


from sentence_transformers import SentenceTransformer, CrossEncoder

import ir_datasets

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tqdm.auto import tqdm # Ensure tqdm is imported for progress bars

import logging
import sys
import os
from IPython.display import display, Markdown

import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"running on {device}")

## Load and Prepare the Dataset

For the examples shown here will will use the MS MARCO dataset. The MS MARCO dataset is widely used for testing and evaluation of information retrieval algorithms. This dataset contains short queries and longer passages. The goal is to search the passages to retrieve the best response for the query. There is a binary indicator showing the passages that are ranked as best for the queries.    

You can find more information on the many variations of the MS MARCO datasets [here](https://microsoft.github.io/msmarco/Datasets.html) and [here](https://sbert.net/examples/sentence_transformer/training/ms_marco/README.html). Alternative downloads of the MS MARCO datasets are available in the [`ir_datasets1` package](https://ir-datasets.com/msmarco-passage.html) or [`SentenceTransformers` package](https://sbert.net/examples/sentence_transformer/training/ms_marco/README.html).   

To start, execute the code in the cell below to load the dataset and display the storage format.

In [ ]:
from datasets import load_dataset

# MS MARCO v2.1 contains ~1M queries (train split)
ds = load_dataset("microsoft/ms_marco", "v2.1")

print(ds)
dataset = ds["train"]

You can see that the dataset is structured as a dictionary of dictionaries. The top level dictionary contains validation, train and test datasets. Within each set there is a list containing answers, passages, query, query id, query type and an indicator if the query is well-formed.

The entire MS MARCO benchmark dataset is quite large, requiring large-scale storage and computing capabilities. To reduce the data size we are working with for this notebook, we are only using the training dataset.  

## Preparing the Dataset

In this section of the notebook, the dataset is prepared for search.    

### Extracting queries and passages    

Our goal is the use approximate nearest neighbor search to find the most relevant passages for each of the queries using approximate nearest neighbor search methods. To prepare for embedding, the queries and passages must be extracted from the dataset. The code in the cell below creates two dictionaries, one for the passages and one for the queries. Along with the text, an identifier mapping the query to the relevant passage or vice versa is added, using the dictionary keys or ids of the other dictionary. Queries with no relevant passage are filtered out. Execute this code and examine the results.

In [ ]:
queries_dict = {}
# The number of queries is the number of rows in the dataset.
print(f"Expected number of queries: {len(dataset)}")
print("Extracting all queries...")
# Iterate directly over the Dataset object to get each query's data
for item in tqdm(dataset, desc="Extracting queries"):
    queries_dict[str(item['query_id'])] = item['query'] # Ensure query_id in queries_dict is string

passages_dict = {} # Maps generated_p_id -> p_text
passage_text_to_id_map = {} # NEW: Maps p_text -> generated_p_id for efficient reverse lookup

print("Extracting all unique passages...")

# Using a set to track seen passage texts to ensure uniqueness
seen_passage_texts = set()
passage_id_counter = 0

# Iterate through each query in the dataset
for item in tqdm(dataset, desc="Collecting passages from all queries"):
    # Each item['passages'] is a dict like {'is_selected': [...], 'passage_text': [...], 'url': [...]}
    if 'passages' in item and item['passages'] is not None:
        passage_texts_for_query = item['passages']['passage_text']

        for p_text in passage_texts_for_query:
            if p_text not in seen_passage_texts:
                # Generate a unique ID for each unique passage text
                generated_p_id = f"p_{passage_id_counter}"
                passages_dict[generated_p_id] = p_text
                passage_text_to_id_map[p_text] = generated_p_id # Populate reverse map
                seen_passage_texts.add(p_text)
                passage_id_counter += 1

print(f"Extracted {len(queries_dict)} queries.")
print(f"Extracted {len(passages_dict)} unique passages.")

### Subsample dataset

The test dataset is still quite large. The code in the cell below subsamples 10% of the queries and along with the corresponding relevant passages. Passages not relevant to any query are also sampled to bring the total number of queries to 300000, creating a larger search space. This process reduces size allows much faster execution and still demonstrates the principles of this lesson.   

As a first set, a query to passage map is built by the code in the cell below. As queries are sampled, the map is used to ensure the relevant passage is also sampled. Execute this code.

In [ ]:
# Build a map from query_id (int) to its corresponding item in the dataset for efficient lookup.
# This map is crucial for efficient subsampling that includes relevant passages.
print("Building query_id to item map from the full dataset for efficient lookup...")
query_id_to_item_map = {item['query_id']: item for item in tqdm(dataset, desc="Mapping query IDs to dataset items")}

The code in the cell below preforms the random sampling on the queries. The relevant passages are then sampled. Additional passages are then random sampled to build the size of the search set. Execute this code.   

In [ ]:
## Set the sample limits
fraction = 0.1
target_total_passages = 300000

print("Subsampling " + str(fraction * 100) +"% of queries (with guaranteed relevant passages) and collecting relevant/non-relevant passages...")

# Ensure the queries have at least one relevant passage
queries_with_relevant_passages = []
for query_id_str, query_text in tqdm(queries_dict.items(), desc="Identifying queries with relevant passages"):
    query_id_int = int(query_id_str)
    item = query_id_to_item_map.get(query_id_int)

    found_relevant_for_this_query = False
    if item and 'passages' in item and item['passages'] is not None:
        is_selected = item['passages']['is_selected']
        if 1 in is_selected: # Check if any passage for this query is marked as relevant
            found_relevant_for_this_query = True

    if found_relevant_for_this_query:
        queries_with_relevant_passages.append(query_id_str)

print(f"Found {len(queries_with_relevant_passages)} queries with at least one relevant passage.")

# Subsample these filtered queries
sampled_query_ids = random.sample(queries_with_relevant_passages, int(len(queries_dict) * fraction))
subsampled_queries_dict = {q_id: queries_dict[q_id] for q_id in sampled_query_ids}

subsampled_passages_dict = {}
subsampled_passage_text_to_id_map = {}
subsampled_passage_id_counter = 0 # New counter for subsampled passages

# Collect all relevant passages for the subsampled queries
for query_id_str in tqdm(sampled_query_ids, desc="Collecting relevant passages for subsampled queries"):
    query_id_int = int(query_id_str)
    item = query_id_to_item_map.get(query_id_int)

    if item and 'passages' in item and item['passages'] is not None:
        passage_texts = item['passages']['passage_text']
        is_selected = item['passages']['is_selected']

        for p_text, selected in zip(passage_texts, is_selected):
            if selected == 1:
                # Add to subsampled_passages_dict if not already present
                if p_text not in subsampled_passage_text_to_id_map:
                    generated_p_id = f"p_sub_{subsampled_passage_id_counter}" # Use a distinct prefix for subsampled IDs
                    subsampled_passages_dict[generated_p_id] = p_text
                    subsampled_passage_text_to_id_map[p_text] = generated_p_id
                    subsampled_passage_id_counter += 1



# Random sample other passages to reach target_total_passages
current_passage_count = len(subsampled_passages_dict)

if current_passage_count < target_total_passages:
    num_additional_needed = target_total_passages - current_passage_count

    # Identify candidate non-relevant passages from the *full* passages_dict
    # These are passages whose text is NOT already in our subsampled set
    candidate_non_relevant_texts = []
    for p_id, p_text in passages_dict.items(): # Iterate over the full passages_dict
        if p_text not in subsampled_passage_text_to_id_map:
            candidate_non_relevant_texts.append(p_text)

    # Randomly select additional non-relevant passages
    if len(candidate_non_relevant_texts) > 0:
        num_to_sample = min(num_additional_needed, len(candidate_non_relevant_texts))
        additional_non_relevant_texts = random.sample(candidate_non_relevant_texts, num_to_sample)

        # Add these selected non-relevant texts to our subsampled dictionaries
        for p_text in additional_non_relevant_texts:
            if p_text not in subsampled_passage_text_to_id_map: # Double check to prevent duplicates if by chance
                generated_p_id = f"p_sub_{subsampled_passage_id_counter}"
                subsampled_passages_dict[generated_p_id] = p_text
                subsampled_passage_text_to_id_map[p_text] = generated_p_id
                subsampled_passage_id_counter += 1

print(f"Subsampled {len(subsampled_queries_dict)} queries.")
print(f"Subsampled {len(subsampled_passages_dict)} total passages (including relevant and non-relevant).")

### Explore the passages and queries      

It is always helpful to have a feel for the nature of the data one is working with. The code in the cell below samples a few queries. The queries, relevant passage and relevance score for the passage are then displayed. Execute the code and examine the results.

In [ ]:
# The query_id_to_item_map has already been built after af8cae33 and before subsampling.
# We will use the subsampled dictionaries for displaying relevant passages.

# Get the first 4 query IDs from the subsampled dictionary (these are strings)
first_n_query_ids_subsampled = list(subsampled_queries_dict.keys())[:4]

# Iterate through the first N queries and find their relevant passages
print(f"\nDisplaying information for the first {len(first_n_query_ids_subsampled)} subsampled queries:\n")

for query_id_str in first_n_query_ids_subsampled: # query_id_str is like '1185869'
    query_id_int = int(query_id_str) # Convert to int for lookup in query_id_to_item_map
    query_text = subsampled_queries_dict[query_id_str]
    display(Markdown(f"**Query ID:** {query_id_str}"))
    display(Markdown(f"**Query Text:** {query_text}"))

    found_passages_for_query = 0

    # Efficiently get the item from the full dataset using the pre-built map
    item = query_id_to_item_map.get(query_id_int)

    if item:
        if 'passages' in item and item['passages'] is not None:
            passage_texts = item['passages']['passage_text']
            is_selected = item['passages']['is_selected']

            for p_text, selected in zip(passage_texts, is_selected):
                if selected == 1 and found_passages_for_query < 10: # Limit to 10 relevant passages for brevity
                    # Use the pre-built subsampled_passage_text_to_id_map for efficient lookup
                    matched_p_id = subsampled_passage_text_to_id_map.get(p_text)

                    if matched_p_id:
                        display(Markdown(f"- **Passage ID:** {matched_p_id}"))
                        display(Markdown(f"  **Passage Text:** {textwrap.fill(p_text, width=100)}"))
                        display(Markdown(f"  **Relevance Score:** {selected}"))
                        found_passages_for_query += 1
                    else:
                        # This means a relevant passage text from the dataset was not found in our unique subsampled_passages_dict.
                        print(f"WARNING: Relevant passage text from dataset not found in subsampled_passage_text_to_id_map for query_id {query_id_str}: {p_text[:100]}...")

    else:
        print(f"WARNING: No item found in query_id_to_item_map for query ID: {query_id_int}. This query might not exist in the dataset.")

    if found_passages_for_query == 0:
        display(Markdown("  *No relevant passages with relevance score 1 found in qrels for this query.* "))
    display(Markdown("--- "))

## Preprocess the text in the queries and passages

Before embedding the text of the query and passages we need to perform some standard preprocessing.      
1. All alphabetic characters are set to lower case.   
2. Punctuation and numeric characters are removed.  
3. Common words with no particular semantic meaning, or stop words, are removed.   

Execute the code to preprocess the queries and passages.   

In [ ]:
def preprocess_text(text_list, desc="Preprocessing text"):
    """
    Preprocesses a list of text strings for use with Sentence Transformers.
    Optimized for speed by pre-compiling regex and using tqdm for progress.

    Args:
        text_list (list): A list of strings, where each string is a sentence or text segment.
        desc (str): Description for the tqdm progress bar.

    Returns:
        list: A list of preprocessed strings.
    """
    preprocessed_texts = []
    stop_words = set(stopwords.words('english'))
    # Compile regex for punctuation and digits once outside the loop
    punct_and_digits_pattern = re.compile(f"[{re.escape(string.punctuation)}\\d+]")

    for text in tqdm(text_list, desc=desc):
        # Lowercasing
        text = text.lower()

        # Remove punctuation and numbers in one pass, replacing with space
        text = punct_and_digits_pattern.sub(" ", text)

        # Tokenization to words
        words = word_tokenize(text)

        # Stop word removal and filter out any empty strings resulting from tokenization after removal
        words = [word for word in words if word not in stop_words and word.strip()]

        # Join words and remove any leading/trailing whitespace
        preprocessed_text = ' '.join(words).strip()
        preprocessed_texts.append(preprocessed_text)

    return preprocessed_texts


## Make sure we have the nltk stop words and punkt tokenizer downloaded
try:
    stopwords.words('english')
    nltk.data.find('tokenizers/punkt') # Check if punkt is downloaded
    nltk.data.find('tokenizers/punkt_tab') # Check if punkt_tab is downloaded
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('punkt_tab')


# Preprocess queries and passages from the subsampled datasets
%time preprocessed_queries = preprocess_text(list(subsampled_queries_dict.values()), desc="Preprocessing subsampled queries")
%time preprocessed_passages = preprocess_text(list(subsampled_passages_dict.values()), desc="Preprocessing subsampled passages")

## Build and Evaluate the Dense ANNS       

With the passage text prepared we can build an index to perform dense approximate nearest neighbor search (ANNS).       

> **Exercise 6-1:** One might well ask, why not use an LLM to perform the document search? There are two significant reasons. One reason is the slow query speed of the LLM, which can lead to significant lag is a great number of queries must be process. The other reason is cost. Consider a use case for a small scale business application with the following parameters:
>   - $10^6$ shards of 500 tokens each are uploaded (updated) daily to the corpus.
>   - $10^6$ queries per day with average query length of 500 tokens.
>   - Generated results of 500 tokens per query.
>   - Cost to upload 1 million tokens is $1.00$.        
>   - Cost to generate 1 million tokens $1.00$.
> 1. What is the daily cost for this system?
> 2. What is the annual cost for 250 business days per year?
> 3. For a medium size business unit with revenue of $10 million per year how significant is this cost?         

> **Answer:**
>   - The daily upload cost         
>   - The daily generation cost    
> 1. Daily cost    
> 2. Annual cost    
> 3. 

### Embed the text      

As a first step, the dense text embedding of the passages computed. The dense embedding vectors are used to build the index.

The embedding is performed with a SentenceTransformer model. Execute the code in the cell below to download a smaller version of the SentenceTransformer model, ['all-MiniLM-L6-v2'](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). This model has an input token limit of 256. Input is truncated past this limit.   

Execute the code to embed the passages.   

In [ ]:
# Load the SentenceTransformer model
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name, device=device)

Next execute the code in the cell below to compute the dense embedding vectors for the passages.  

In [ ]:

# Ensure query_embeddings are in the correct format for FAISS (numpy float32)
%time query_embeddings = model.encode(preprocessed_queries, convert_to_tensor=True, show_progress_bar=True).cpu().numpy()

#query_embeddings.astype('float32')

## L2 normalize the embedding vectors
faiss.normalize_L2(query_embeddings)


# Clear CUDA cache before encoding passages to free up GPU memory
torch.cuda.empty_cache()

# Passages are much larger, so we need to process them in batches to avoid OutOfMemoryError
# A batch_size of 256 or 512 is often a good starting point for MiniLM models on Colab GPUs.
# Given the persistent OOM, significantly reducing batch_size for passages.
%time passage_embeddings = model.encode(preprocessed_passages, convert_to_tensor=True, show_progress_bar=True, batch_size=32).cpu().numpy()

## L2 normalize the embedding vectors
faiss.normalize_L2(passage_embeddings)

print(f"Shape of query embeddings: {query_embeddings.shape}")
print(f"Shape of passage embeddings: {passage_embeddings.shape}")

> **Exercise 6-2:** Examine the properties of the embedding model and answer these questions:
> 1. Consider the effect of the input token limit  How will this limitation affect the embeddings of long passages and how can this effect limit the semantics captured and recall of the similarity search?
> 2. How would sharding the long passages help this problem?
> 3. Why is it important that the dimensions of the embedding vector computed are orthogonal.    
> 4. It should be the case that a longer embedding vector will contain better semmantic information, improving recall. What are the tradeoffs of doubling the embedding vector length?  

> **Answers:**
> 1.     
> 2.    
> 3.
> 4. 

## Build and Evaluate the ANNS Index

With the dense embedding vectors computed we are now ready to build an index to perform the ANNS.


### Construct the index      

In this case we will build a composite index with the following stages:
- Othogolalization for the product quantization stage.
- A 4096 cell inverted index course quantizer. 32 probes are applied to this index.
- A hierarchical navigable small world index for finer search.
- Product quantization on 32 subvectors for compression of the index.  


Execute this code to build the index. Execution will take some time. In tests about 20-30 min of wall clock time.   

In [ ]:
def get_memory(index, digits=2):
    # write index to file
    faiss.write_index(index, './temp.index')
    # get file size
    file_size = os.path.getsize('./temp.index')
    # delete saved index
    os.remove('./temp.index')
    return round(file_size/1000000, digits)

D_sop = passage_embeddings.shape[1] # Dimension of SOP image embeddings, now 256

# Create the composite index without PCA, as embeddings are already reduced
index_IVFHNSWPQ = faiss.index_factory(D_sop, "OPQ32,IVF4096_HNSW,PQ32")

# Train the index with SOP image embeddings
print('Training the index')
%time index_IVFHNSWPQ.train(passage_embeddings)
# Add the vectors into the indexer
print('\nAdding passages to index')
%time index_IVFHNSWPQ.add(passage_embeddings)

# we increase nprobe
index_ = faiss.extract_index_ivf(index_IVFHNSWPQ)
index_.nprobe = 32

print(f'Index size: {get_memory(index_IVFHNSWPQ)} MB')

> **Exercise 6-3:**
> 1. We are using the typical choice of **cosine similarity** between natural language embedding vectors. However, you will notice that L2 or dot product, the default metric for FAISS is used to specify the index. What step in the computation of the similarity search transforms to metric to cosine similarity and why? *Hint*, the key step occurs before the foregoing code cell.
> 2. Consider the memory requirements for the data used to build the index and the compression achieved by the index. Assume that the average passage comprises 300 characters in the passage corpus of 300000. The embedding vectors use 4 byte floating point numbers.    
>    a. Compute the size of the raw corpus in bytes.
>    b. Compute the size of the embedding in bytes.    
>    c. Compute compression of achieved by the index with respect to the raw corpus.
>    d. Compute compression of achieved by the index with respect to the embedding.       

> **Answer:**
> 1.     
> 2. Put calculations here:     
>    a. Size of raw corpus      
>    b. Size of embeddings       
>    c. Compression with respect to corpus    
>    d. Compression with respect to embedding      

### Query the index and evaluate

The next step is to query the index and evaluate the result.   

The code in the cell below defines a function for evaluation of the query results. The function computes meanr ecall, mean precision, mean reciprocal rank (MRR), and discounted cumulative gain (DCG) and different values of k and displays them in a table.

> **Note:** Since there is generally only one relevent passage for each query, only mean precision at 1 is actually meaningful. Ignore mean precision at other values of k.  

Execute the code.  

In [ ]:
def calculate_metrics_at_k(retrieved_results, ground_truth_relevant_passages, ks_to_evaluate, subsampled_queries_dict, evaluation_name="Retrieval"):
    """
    Calculates Recall@K, Precision@K, NDCG@K, and MRR@K for given retrieval results against ground truth,
    and also computes overall Mean Reciprocal Rank (MRR). Displays results in a table.

    Args:
        retrieved_results (dict): Dictionary of query_id_str to list of retrieved passage IDs (ranked).
        ground_truth_relevant_passages (dict): Dictionary of query_id_str to list of relevant passage IDs.
        ks_to_evaluate (list): List of integers representing k values for metric calculation.
        subsampled_queries_dict (dict): Dictionary of subsampled queries, used for iterating through queries.
        evaluation_name (str): Name of the evaluation (e.g., "Dense Retrieval") for print statements.

    Returns:
        dict: A dictionary containing the average Recall@k, Precision@k, NDCG@k, MRR@k for each k,
              and the overall MRR.
    """
    k_metrics = {k: {'recall': [], 'precision': [], 'ndcg': [], 'mrr_at_k': []} for k in ks_to_evaluate}
    mrr_per_query_overall = [] # For traditional, overall MRR

    for query_id_str in tqdm(subsampled_queries_dict.keys(), desc=f"Calculating {evaluation_name} Metrics"):
        relevant_docs_set = set(ground_truth_relevant_passages.get(query_id_str, []))
        retrieved_docs_list = retrieved_results.get(query_id_str, [])

        # Calculate overall MRR for the current query (based on first relevant document in entire list)
        query_mrr_overall = 0.0
        for rank, doc_id in enumerate(retrieved_docs_list, 1):
            if doc_id in relevant_docs_set:
                query_mrr_overall = 1.0 / rank
                break
        mrr_per_query_overall.append(query_mrr_overall)

        # If there are no relevant documents for this query, all metrics at k are 0.
        if not relevant_docs_set:
            for k in ks_to_evaluate:
                k_metrics[k]['recall'].append(0.0)
                k_metrics[k]['precision'].append(0.0)
                k_metrics[k]['ndcg'].append(0.0)
                k_metrics[k]['mrr_at_k'].append(0.0)
            continue

        for k in ks_to_evaluate:
            retrieved_at_k = retrieved_docs_list[:k]
            num_hits_at_k = len(set(retrieved_at_k) & relevant_docs_set)

            # Recall@K
            recall_at_k = num_hits_at_k / len(relevant_docs_set)
            k_metrics[k]['recall'].append(recall_at_k)

            # Precision@K
            precision_at_k = num_hits_at_k / k if k > 0 else 0.0
            k_metrics[k]['precision'].append(precision_at_k)

            # DCG@K
            dcg_at_k = 0.0
            for rank_idx, doc_id in enumerate(retrieved_at_k):
                relevance = 1 if doc_id in relevant_docs_set else 0
                # Using 0-indexed rank_idx, so +2 for log base 2 (log2(1) = 0)
                dcg_at_k += relevance / np.log2(rank_idx + 2)

            idcg_at_k = 0.0
            # Ideal DCG assumes all relevant documents appear at the top
            num_ideal_relevant = min(len(relevant_docs_set), k) # Number of actual relevant docs up to k
            for rank_idx in range(num_ideal_relevant):
                idcg_at_k += 1 / np.log2(rank_idx + 2)

            ndcg_at_k = dcg_at_k / idcg_at_k if idcg_at_k > 0 else 0.0
            k_metrics[k]['ndcg'].append(ndcg_at_k)

            # MRR@K
            mrr_at_k_for_query = 0.0
            for rank, doc_id in enumerate(retrieved_at_k, 1):
                if doc_id in relevant_docs_set:
                    mrr_at_k_for_query = 1.0 / rank
                    break
            k_metrics[k]['mrr_at_k'].append(mrr_at_k_for_query)

    # Aggregate results for display
    results_data = []
    for k in ks_to_evaluate:
        avg_recall = np.mean(k_metrics[k]['recall']) if k_metrics[k]['recall'] else 0.0
        avg_precision = np.mean(k_metrics[k]['precision']) if k_metrics[k]['precision'] else 0.0
        avg_ndcg = np.mean(k_metrics[k]['ndcg']) if k_metrics[k]['ndcg'] else 0.0
        avg_mrr_at_k = np.mean(k_metrics[k]['mrr_at_k']) if k_metrics[k]['mrr_at_k'] else 0.0
        results_data.append({
            'k': k,
            'Recall@k': avg_recall,
            'Precision@k': avg_precision,
            'DCG@k': avg_ndcg,
            'MRR@k': avg_mrr_at_k
        })

    # Overall MRR (traditional definition)
    overall_mrr = np.mean(mrr_per_query_overall) if mrr_per_query_overall else 0.0

    print(f"\n--- {evaluation_name} Metrics ---")
    df_metrics = pd.DataFrame(results_data).set_index('k')
    print(df_metrics.to_string(float_format="%.4f")) # Changed to simple print
    print(f"Overall Mean Reciprocal Rank (MRR): {overall_mrr:.4f}")

    # Return a dictionary of all computed metrics for potential further use
    return {
        'k_metrics': {k: {
            'recall': np.mean(k_metrics[k]['recall']) if k_metrics[k]['recall'] else 0.0,
            'precision': np.mean(k_metrics[k]['precision']) if k_metrics[k]['precision'] else 0.0,
            'ndcg': np.mean(k_metrics[k]['ndcg']) if k_metrics[k]['ndcg'] else 0.0,
            'mrr_at_k': np.mean(k_metrics[k]['mrr_at_k']) if k_metrics[k]['mrr_at_k'] else 0.0
        } for k in ks_to_evaluate},
        'mrr_overall': overall_mrr
    }

The code in the cell below executes and evaluates the results of the queries on the index by the following steps:    
1. The ground truth is extracted.    
2. The queries are embedded.      
3. The nearest neighbor passages are found for each of the queries.
4. The evaluation function is called.

Execute the code.

In [ ]:
def perform_dense_retrieval_and_evaluate(
    subsampled_passages_dict,
    subsampled_queries_dict,
    query_id_to_item_map,
    subsampled_passage_text_to_id_map,
    query_embeddings,
    index_IVFHNSWPQ,
    # Removed calculate_metrics_at_k from function arguments
    evaluation_name="Dense Retrieval"
):
    # Map FAISS indices back to subsampled passage IDs
    subsampled_passage_ids_list = list(subsampled_passages_dict.keys())

    # Prepare ground truth relevant passages for evaluation
    ground_truth_relevant_passages = {}
    for query_id_str, query_text in tqdm(subsampled_queries_dict.items(), desc="Preparing ground truth"):
        query_id_int = int(query_id_str)
        item = query_id_to_item_map.get(query_id_int)
        relevant_p_ids_for_query = []

        if item and 'passages' in item and item['passages'] is not None:
            passage_texts = item['passages']['passage_text']
            is_selected = item['passages']['is_selected']

            for p_text, selected in zip(passage_texts, is_selected):
                if selected == 1:
                    matched_p_id = subsampled_passage_text_to_id_map.get(p_text)
                    if matched_p_id:
                        relevant_p_ids_for_query.append(matched_p_id)
        ground_truth_relevant_passages[query_id_str] = relevant_p_ids_for_query

    # Perform queries and collect results
    ks_to_evaluate = [1, 2, 10, 20, 100]
    k_max = max(ks_to_evaluate) # Retrieve up to the maximum k needed

    retrieved_results = {}

    # Iterate through queries to perform search
    for i, query_id_str in tqdm(enumerate(subsampled_queries_dict.keys()), total=len(subsampled_queries_dict), desc="Performing FAISS search"):
        query_emb = query_embeddings[i].reshape(1, -1)
        # D: distances, I: indices of retrieved passages
        D, I = index_IVFHNSWPQ.search(query_emb, k_max)

        # Map FAISS indices (I) back to our subsampled passage IDs
        retrieved_p_ids = [subsampled_passage_ids_list[idx] for idx in I[0] if idx != -1] # Filter out -1 for non-found
        retrieved_results[query_id_str] = retrieved_p_ids

    # Return all necessary data for evaluation outside the function
    return retrieved_results, ground_truth_relevant_passages, ks_to_evaluate, k_max # Return k_max as well

%time dense_retrieved_results, dense_ground_truth_relevant_passages, dense_ks_to_evaluate, dense_k_max = perform_dense_retrieval_and_evaluate( \
    subsampled_passages_dict, \
    subsampled_queries_dict, \
    query_id_to_item_map, \
    subsampled_passage_text_to_id_map, \
    query_embeddings, \
    index_IVFHNSWPQ, \
    evaluation_name="Dense Retrieval" \
)

# Evaluate after the function has executed
dense_recall_scores = calculate_metrics_at_k(
    dense_retrieved_results,
    dense_ground_truth_relevant_passages,
    dense_ks_to_evaluate,
    subsampled_queries_dict,
    evaluation_name="Dense Retrieval"
)

> **Exercise 6-4:** Explain why the NSCG and MRR must increase with k.               

> **Answer:**     .    

## Sparse Search   

The sparse search is an alternative to dense search. Sparse search uses a sparse embedding which has specific representation for each token or word in the vocabulary.   

The code in the cell below does the following:     
1. The passages are tokenized. Tokenization is essential for sparse representations since each token in the vocabulary is represented.       
2. The sparse receiver is constructed, which is a sparse data structure to hold the vocabulary in an inverted index.
3. The tokens are added to the receiver with the computed token weights.

Execute this code.      

> **Note on scaling:** Originally this notebook was intended to operate on a 20% subsample of the MS MARCO dataset. On Colab, the scaling limitations of BM25 forced the decision to use a 10% sample. The runtime and memory use by the BM25 algorithm proved difficult to manage.   

In [ ]:
print('Tokenizing the corpus')
%time corpus_tokens = bm25s.tokenize(preprocessed_passages, stopwords="en",)
print('Building the retriever')
%time sparse_retriever = bm25s.BM25()
print('Adding tokens to the retriever')
%time sparse_retriever.index(corpus_tokens)

We are now ready to test the sparse BM25 receiver model. The code in the cell below does the following:      
1. The ground truth results are prepared.
2. The queries are tokenized.
3. The similarity search is performed on the sparse retriever.
4. The results of the search are evaluated.

Execute this code.   

In [ ]:
# Map sparse retriever indices back to subsampled passage IDs
subsampled_passage_ids_list_sparse = list(subsampled_passages_dict.keys())

# Prepare ground truth relevant passages for evaluation (re-using from dense retrieval)
ground_truth_relevant_passages_sparse = dense_ground_truth_relevant_passages # Already computed in da9b29bc

# Perform sparse queries and collect results
ks_to_evaluate_sparse = [1, 2, 10, 20, 100]
k_max_sparse = max(ks_to_evaluate_sparse) # Retrieve up to the maximum k needed

# Tokenize the queries in a single batch
all_query_ids = list(subsampled_queries_dict.keys())
all_query_texts = [subsampled_queries_dict[q_id] for q_id in all_query_ids] # Get texts in order of IDs

print(f"Tokenizing {len(all_query_texts)} queries in batch...")
batch_query_tokens = bm25s.tokenize(all_query_texts, stopwords="en")

# Perform batch retrieval for the queries
print("Performing BM25 batch retrieval...")
# sparse_retriever.retrieve returns two NumPy arrays: doc_ids and scores
# doc_ids will be a 2D array [num_queries, k_max_sparse]
# scores will be a 2D array [num_queries, k_max_sparse]
%time batch_doc_ids_from_bm25, batch_bm25_scores = sparse_retriever.retrieve(batch_query_tokens, k=k_max_sparse)

# Subsample the queries for evaluation to avoid resource limits
eval_sample_size = 100 # Evaluate metrics for a sample of 100 queries

# Ensure there are enough queries to sample, otherwise use all available
actual_eval_sample_size = min(eval_sample_size, len(all_query_ids)) # Use all_query_ids for sampling
eval_query_ids = random.sample(all_query_ids, actual_eval_sample_size)

# Create a mapping from query_id_str to its original index in the `all_query_ids` list
query_id_to_batch_idx = {q_id: i for i, q_id in enumerate(all_query_ids)}

# Process results only for sampled queries
print("Processing batch retrieval results for sampled queries...")
sampled_sparse_retrieved_results = {} # This will now only store sampled queries' results

for query_id_str in tqdm(eval_query_ids, desc="Mapping BM25 results to passage IDs for sampled queries"):
    # Get the index of the current sampled query in the original batch
    batch_idx = query_id_to_batch_idx[query_id_str]

    # Use this index to retrieve the doc_ids from the batch results
    current_query_doc_ids = batch_doc_ids_from_bm25[batch_idx]

    # Ensure indices are valid before mapping (filter out -1 if present)
    retrieved_p_ids_sparse = [subsampled_passage_ids_list_sparse[idx] for idx in current_query_doc_ids if idx != -1]
    sampled_sparse_retrieved_results[query_id_str] = retrieved_p_ids_sparse


# Create reduced dicts for evaluation (these are already correct from previous iteration)
subsampled_queries_dict_for_eval = {q_id: subsampled_queries_dict[q_id] for q_id in eval_query_ids}
sampled_ground_truth_relevant_passages_sparse = {q_id: ground_truth_relevant_passages_sparse[q_id] for q_id in eval_query_ids}

# Call the new function for the sampled queries
sparse_recall_scores = calculate_metrics_at_k(
    sampled_sparse_retrieved_results,
    sampled_ground_truth_relevant_passages_sparse,
    ks_to_evaluate_sparse,
    subsampled_queries_dict_for_eval,
    evaluation_name=f"Sparse Retrieval (sampled {actual_eval_sample_size} queries)"
)

When interpreting these results, keep in mind that the MS MACRO dataset typically has only one relevant passage for each query. 

> **Exercise 6-5:** Examine the results computed with the sparse receiver and answer these questions.
> 1. Compare the recall, DCG and MRR of the Sparse retriever to the dense retriever. What does the difference in performance tell you about the importance of key words vs. semantics for this task? Do you think this comparison generalizes to all IR tasks regardless of the nature of the passages and queries?
> 2. Compare the CPU `user` times for the FAISS search to the BM25 receiver search. It is reasonable to assume that dense vector search and and BM25s algorithm scale as approximately $log(n)$. Explain how the differences in performance will scale for these two approaches.       
> 3. An alternative to using the BM25 algorithm is to compute sparse embeddings with a more sophisticated algorithm like SPLADE. What are the advantages and disadvantages of such an approach compared to the one applied here? What is the most significant disadvantage of using the SPLADE algorithm.      

> **Answers:**
> 1.         
> 2.     
> 3.      

## Hybrid Retrieval and Evaluation

Hybrid rankers combine the results from both the dense (FAISS) and sparse (BM25) retrievers to create a hybrid ranking. The scores from both models are normalized and then summed to produce a combined score for each passage. The combined score is weighted to favor either retriever. Higher weight on dense retrieval favors semantic similarity. Whereas, higher weight on sparse retrieval favors keyword similarity. The optimal weighting is task-specific. For simplicity and the purpose of demonstration we use equal weighting of scores in this example.     

To sum scores, the distance computed for the dense retriever must be converted to a similarity score.     
$$similarity\ score = \frac{1.0}{1.0 + distance}$$

The code in the cell below computes and ranks the hybrid scores and computes and displays the metrics. Execute the code.  

In [ ]:
# Define the hyperparameter for dense retriever weight (0.0 to 1.0)
dense_weight = 0.5 # Example: 0.5 gives equal weight, 1.0 relies only on dense, 0.0 relies only on sparse
sparse_weight = 1.0 - dense_weight

# Extract Dense Scores for all queries
print("Extracting dense retrieval scores...")
dense_results_with_scores = {}

# Define subsampled_passage_ids_list here, as it's used for dense retrieval mapping
subsampled_passage_ids_list = list(subsampled_passages_dict.keys())

# Ensure query_embeddings are in the correct format for FAISS (numpy float32)
query_embeddings_faiss = query_embeddings.astype('float32')

# Iterate through queries to perform search
for i, query_id_str in tqdm(enumerate(subsampled_queries_dict.keys()), total=len(subsampled_queries_dict), desc="Performing FAISS search for scores"):
    query_emb = query_embeddings_faiss[i].reshape(1, -1)
    D, I = index_IVFHNSWPQ.search(query_emb, dense_k_max) # Use dense_k_max now

    # Map FAISS indices (I) back to the subsampled passage IDs and convert distances to similarity scores
    # Smaller distance is better, so 1 / (1 + distance) converts to a similarity score where higher is better.
    dense_scores_for_query = []
    for idx, dist in zip(I[0], D[0]):
        if idx != -1:
            p_id = subsampled_passage_ids_list[idx]
            # Convert distance to similarity score; add 1 to avoid division by zero and handle large distances
            similarity_score = 1.0 / (1.0 + dist)
            dense_scores_for_query.append((p_id, similarity_score))
    dense_results_with_scores[query_id_str] = dense_scores_for_query

# --- 2. Extract Sparse Scores for all queries ---
print("Extracting sparse retrieval scores...")
sparse_results_with_scores = {}

# We already have batch_doc_ids_from_bm25 and batch_bm25_scores from cell 99v2yXOt4Gbc
# We need to map them back to query_id_str

for i, query_id_str in tqdm(enumerate(subsampled_queries_dict.keys()), total=len(subsampled_queries_dict), desc="Extracting BM25 scores"):
    bm25_scores_for_query = []
    current_doc_ids = batch_doc_ids_from_bm25[i] # NumPy array of doc_ids for current query
    current_scores = batch_bm25_scores[i]       # NumPy array of scores for current query

    for idx_in_batch, score in zip(current_doc_ids, current_scores):
        if idx_in_batch != -1:
            p_id = subsampled_passage_ids_list_sparse[idx_in_batch]
            bm25_scores_for_query.append((p_id, score))
    sparse_results_with_scores[query_id_str] = bm25_scores_for_query

# --- 3. Combine and Re-rank Results ---
print("Combining and re-ranking hybrid results...")
hybrid_retrieved_results = {}

# Determine the maximum k from both dense and sparse evaluations
combined_k_max = max(dense_k_max, k_max_sparse)

for query_id_str in tqdm(subsampled_queries_dict.keys(), desc="Creating hybrid ranking"):
    combined_passage_scores = {}

    # Add dense scores with weight
    for p_id, score in dense_results_with_scores.get(query_id_str, []):
        combined_passage_scores[p_id] = combined_passage_scores.get(p_id, 0.0) + (score * dense_weight)

    # Add sparse scores with weight
    for p_id, score in sparse_results_with_scores.get(query_id_str, []):
        # For BM25, a higher score is better, so directly add it
        combined_passage_scores[p_id] = combined_passage_scores.get(p_id, 0.0) + (score * sparse_weight)

    # Sort passages by combined score in descending order
    sorted_passages = sorted(combined_passage_scores.items(), key=lambda item: item[1], reverse=True)

    # Take top `combined_k_max` passage IDs
    hybrid_retrieved_results[query_id_str] = [p_id for p_id, _ in sorted_passages[:combined_k_max]]

# Call the new function
hybrid_recall_scores = calculate_metrics_at_k(hybrid_retrieved_results, dense_ground_truth_relevant_passages, ks_to_evaluate_sparse, subsampled_queries_dict, evaluation_name="Hybrid Retrieval")

When interpreting these results, keep in mind that the MS MACRO dataset typically has only one relevant passage for each query.    

> **Exercise 6-6:** Examine the results computed with the hybrid receiver and answer these questions.
> 1. Compare the recall, DCC and MRR of the hybrid retriever to the results of the stand alone FAISS dense search and the BM25 sparse search. 
> 2. What does this change in the performance metrics tell you about the independence of the errors of the two rankers?       

> **Answers:**
> 1.      
> 2.      

## Reranking  

Now that we have an initial ranking from the hybrid ranker, we can try to improve the result by using **reranker**. In this case we will use a lightweight cross-encoder, ['ms-marco-MiniLM-L6-v2'](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2). The cross encoder uses transformers to compute a cross-attention similarity.    

Execute the code in the cell below to load the pretrained cross encoder model.   

In [ ]:
pretrained_model = "cross-encoder/ms-marco-MiniLM-L6-v2"
model_CrossEncoder = CrossEncoder(pretrained_model)

The code in the cell below executes and evaluates the cross-encoder reranker by the following steps:    
1. Query-passage pairs are organized for the reranking process. The passages have been found by the hybrid retriever.       
2. Cross-attention scores are computed.     
3. Passages are ranked based on the cross-attention scores.
4. The evaluation metrics are computed and displayed.

Execute this code   

> **Note:** To limit the run time the code in the cell below randomly subsamples only 1000 of the total 81 thousand queries. 

In [ ]:
def reranker(queries):
  final_reranked_results = {}
  for query_id_str, passages_with_scores in tqdm(queries.items(), desc="Final sorting for reranked results"):
    # Sort by rerank score in descending order
    sorted_reranked_passages = sorted(passages_with_scores, key=lambda item: item[1], reverse=True)
    # Extract just the passage IDs in the new ranked order
    final_reranked_results[query_id_str] = [p_id for p_id, _ in sorted_reranked_passages]
  return final_reranked_results

def predict_rerank_scores(model_CrossEncoder, all_rerank_pairs):
    if all_rerank_pairs:
        return model_CrossEncoder.predict(all_rerank_pairs, batch_size=512, show_progress_bar=True)
    else:
        return []


print("Optimizing and reranking top 100 hybrid results for a sample of 100 queries using CrossEncoder...")
k_rerank = 100
rerank_sample_size = 1000

# Randomly sample queries for reranking
# Ensure there are enough queries to sample, otherwise use all available
actual_rerank_sample_size = min(rerank_sample_size, len(subsampled_queries_dict))
rerank_query_ids = random.sample(list(subsampled_queries_dict.keys()), actual_rerank_sample_size)

# Create a reduced subsampled_queries_dict for iteration and evaluation
subsampled_queries_dict_for_rerank = {q_id: subsampled_queries_dict[q_id] for q_id in rerank_query_ids}

# Collect all query-passage pairs for batch prediction
all_rerank_pairs = []
query_passage_map = [] # To map scores back to original query and passage IDs

print(f"Collecting query-passage pairs for batch reranking for {actual_rerank_sample_size} queries...")
for query_id_str in tqdm(subsampled_queries_dict_for_rerank.keys(), desc="Collecting pairs"):
    query_text = subsampled_queries_dict_for_rerank[query_id_str]

    # Get the top k_rerank passage IDs from the hybrid results
    top_hybrid_p_ids = hybrid_retrieved_results.get(query_id_str, [])[:k_rerank]

    if not top_hybrid_p_ids:
        # Store empty list for this query, will be handled during score distribution
        query_passage_map.append({'query_id_str': query_id_str, 'pairs_count': 0})
        continue

    passage_texts_for_rerank = [subsampled_passages_dict[p_id] for p_id in top_hybrid_p_ids]

    current_pairs = []
    for p_id, p_text in zip(top_hybrid_p_ids, passage_texts_for_rerank):
        current_pairs.append([query_text, p_text])
        query_passage_map.append({'query_id_str': query_id_str, 'p_id': p_id})

    all_rerank_pairs.extend(current_pairs)

# Perform batch prediction of the reranking scores with the CrossEncoder
print(f"Performing batch prediction for {len(all_rerank_pairs)} query-passage pairs...")
# model_CrossEncoder.predict returns a list of scores (floats)
%time all_rerank_scores = predict_rerank_scores(model_CrossEncoder, all_rerank_pairs)

# Distribute scores and re-rank results for each query
reranked_results = {}
score_idx = 0

print("Distributing scores and re-ranking...")

# Create a temporary list to hold (p_id, score) pairs for each query for efficient sorting
# Initialize only for the sampled queries
query_temp_scores = {query_id_str: [] for query_id_str in subsampled_queries_dict_for_rerank.keys()}

for q_map_entry in tqdm(query_passage_map, desc="Distributing scores"):
    query_id_str = q_map_entry['query_id_str']

    if 'p_id' not in q_map_entry: # This was a query with no hybrid results
        # No need to do anything here, query_temp_scores already has an empty list for it
        continue

    p_id = q_map_entry['p_id']
    score = all_rerank_scores[score_idx]
    score_idx += 1

    query_temp_scores[query_id_str].append((p_id, score))

# Now, for each query, sort the collected (p_id, score) pairs
%time final_reranked_results = reranker(query_temp_scores)
#for query_id_str, passages_with_scores in tqdm(query_temp_scores.items(), desc="Final sorting for reranked results"):
#    # Sort by rerank score in descending order
#    sorted_reranked_passages = sorted(passages_with_scores, key=lambda item: item[1], reverse=True)
#    # Extract just the passage IDs in the new ranked order
#    final_reranked_results[query_id_str] = [p_id for p_id, _ in sorted_reranked_passages]


# Evaluate the reranked results
ks_to_evaluate_rerank = [1, 2, 10, 20, 100]
rerank_recall_scores = calculate_metrics_at_k(
    final_reranked_results,
    dense_ground_truth_relevant_passages,
    ks_to_evaluate_rerank,
    subsampled_queries_dict_for_rerank, # Use the reduced query dict for evaluation
    evaluation_name="CrossEncoder Reranking (100 queries)"
)

> **Exercise 6-7:** Examine these results and answer the following questions:
> 1. Why is the cross encoder a good choice for reranking, but would not be suitable for the initial retrieval step.
> 2. What evidence do you see that the reranking has improved the retrieval results.
> 3. Compare the execution time of the per query of the FAISS retriever to the cross-encoder receiver. What does this difference in performance along with the improved results tell you about why the retriever-receiver (ranker-reranker) architecture is widely used?   

> **Answers:**
> 1.     
> 2.     
> 3.     

Finally, let's have a look at the results of the reranking. The code in the cell below displays 3 randomly sampled queries along with the top 5 passages found for that query.  Keep in mind that the MS MARCO dataset typically only has 1 relevant passage for a given query. Execute this code and examine the results.  

In [ ]:
n_queries = 3
n_passages = 5

print("\n--- Sampled Reranked Results ---")

# Randomly sample 2 query IDs from the queries that were reranked
sampled_rerank_query_ids = random.sample(list(subsampled_queries_dict_for_rerank.keys()), n_queries)

for query_id_str in sampled_rerank_query_ids:
    query_text = subsampled_queries_dict_for_rerank[query_id_str]
    display(Markdown(f"**Query ID:** {query_id_str}"))
    display(Markdown(f"**Query Text:** {query_text}"))

    # Get the top 5 reranked passage IDs for this query
    top_5_reranked_p_ids = final_reranked_results.get(query_id_str, [])[:n_passages]

    if top_5_reranked_p_ids:
        display(Markdown("**Top 5 Reranked Passages:**"))
        for rank, p_id in enumerate(top_5_reranked_p_ids, 1):
            p_text = subsampled_passages_dict.get(p_id, "[Passage Not Found]")
            display(Markdown(f"- **Rank {rank}, Passage ID:** {p_id}"))
            display(Markdown(f"  **Passage Text:** {textwrap.fill(p_text, width=100)}"))
    else:
        display(Markdown("  *No reranked passages found for this query.* "))
    display(Markdown("--- "))

#### Copyright 2025, 2026, Stephen F Elston. All rights reserved.